# Week 1: Recurrent Neural Networks

Discover recurrent neural networks, a type of model that performs extremely well on temporal data, and several of its variants, including LSTMs, GRUs and Bidirectional RNNs.

**Learning Objectives:**

- Define notation for building sequence models
- Describe the architecture of a basic RNN
- Identify the main components of an LSTM
- Implement backpropagation through time for a basic RNN and an LSTM
- Give examples of several types of RNN
- Build a character-level text generation model using an RNN
- Store text data for processing using an RNN
- Sample novel sequences in an RNN
- Explain the vanishing/exploding gradient problem in RNNs
- Apply gradient clipping as a solution for exploding gradients
- Describe the architecture of a GRU
- Use a bidirectional RNN to take information from two points of a sequence
- Stack multiple RNNs on top of each other to create a deep RNN
- Use the flexible Functional API to create complex models
- Generate your own jazz music with deep learning
- Apply an LSTM to a music generation task

---

## Why Sequence Models?

This section introduces **Sequence Models**. These models, primarily **Recurrent Neural Networks (RNNs)**, are designed to handle data where the order of information—the "sequence"—is the most critical feature.

### Key Concepts: What is a Sequence Model?

* **Temporal/Ordered Data:** Unlike standard images where pixels are processed spatially, sequence models process data that unfolds over time or follows a specific linear order (like words in a sentence or bases in DNA).
* **Supervised Learning Framework:** Like previous models, these are trained using $(X, Y)$ pairs, but the nature of $X$ and $Y$ changes significantly depending on the task.

### Diverse Applications of Sequence Models

Sequence problems come in many "shapes" regarding the lengths of input ($T_x$) and output ($T_y$):

| Application | Input ($X$) | Output ($Y$) | Sequence Type |
| --- | --- | --- | --- |
| **Speech Recognition** | Audio Clip | Text Transcript | Many-to-Many ($T_x \neq T_y$) |
| **Music Generation** | Genre/Empty | Audio/Notes | One-to-Many |
| **Sentiment Analysis** | Text Review | Star Rating (1–5) | Many-to-One |
| **DNA Analysis** | A, C, G, T bases | Protein Labels | Many-to-Many ($T_x = T_y$) |
| **Machine Translation** | French Sentence | English Sentence | Many-to-Many ($T_x \neq T_y$) |
| **Video Activity** | Video Frames | Activity Label | Many-to-One |
| **Name Entity Recognition** | Sentence | Identity Labels | Many-to-Many ($T_x = T_y$) |

### Why RNNs are Necessary

Standard Neural Networks (Densely connected) struggle with sequences for two main reasons:

1. **Variable Length:** Standard networks require a fixed input size. Sequences (like sentences) vary in length.
2. **Shared Features:** A word learned at the start of a sentence should have its meaning preserved if it appears at the end. Standard networks don't easily share features across different positions.

### What's Next?

In the bext section, we will define the mathematical notation required to represent these sequences (using $x^{\langle t \rangle}$ to represent the element at time $t$) and how to represent words using **Vocabulary** and **One-Hot Encoding**.

---

## Notation

This section defines the standard **Notation and Data Representation** used in Natural Language Processing (NLP) and sequence models. Using the example of **Named-entity recognition (NER)**, it explains how words are transformed into mathematical structures that a neural network can process.

### Key Concepts: Sequence Notation

To handle sequences, we use specific indexing to track both the position in the sequence and the specific training example:

* **$x^{\langle t \rangle}$**: Represents the $t^{th}$ element in the input sequence (e.g., the $3^{rd}$ word in a sentence).
* **$y^{\langle t \rangle}$**: Represents the $t^{th}$ element in the output sequence.
* **$T_x$ and $T_y$**: Denote the total length of the input and output sequences, respectively. These can differ depending on the task.
* **$x^{(i)\langle t \rangle}$**: Refers to the $t^{th}$ element of the $i^{th}$ training example.
* **$T_x^{(i)}$**: The length of the input sequence for the $i^{th}$ training example (acknowledging that different sentences have different lengths).

### Vocabulary and Word Representation

Before a computer can process a sentence, each word must be converted into a numerical vector.

* **The Vocabulary (Dictionary):** A list of all unique words used in the system.
    * Sizes typically range from **10,000** (for basic tasks) to **100,000+** (for commercial applications).
    * Words are sorted (usually alphabetically) and assigned an index (e.g., "And" = 367, "Harry" = 4075).
* **One-Hot Vectors:** Each word $x^{\langle t \rangle}$ is represented by a vector the size of the vocabulary.
    * The vector contains all zeros except for a **1** at the index corresponding to that word in the dictionary.
    * *Example:* If "Harry" is at index 4075, its one-hot vector is $[0, 0, ..., 1, ..., 0]^T$ with the 1 at position 4075.

### Handling the Unknown Tokens

A common problem in NLP is encountering a word during testing that was not in the training vocabulary.

* **Unknown Word Token (`<UNK>`):** A special "fake word" added to the vocabulary to represent any word not found in the dictionary. This allows the model to still process the rest of the sentence without crashing.

### EXample of NER Representation

In the example sentence *"Harry Potter and Hermione Granger invented a new spell"*:

* $x^{\langle 1 \rangle}$ ("Harry") maps to $y^{\langle 1 \rangle} = 1$ (Person).
* $x^{\langle 3 \rangle}$ ("and") maps to $y^{\langle 3 \rangle} = 0$ (Not a person).
* The model learns to predict these binary labels for each position $t$.

---

## Recurrent Neural Network Model

In this section, the limitations of standard neural networks for sequence data are addressed, leading to the introduction of the **Recurrent Neural Network (RNN)**. The RNN architecture is designed specifically to handle variable-length inputs and share learned features across different positions in a sequence.

### Why Standard Neural Networks Fail for Sequences

Standard (dense) networks struggle with two primary issues:

* **Variable Lengths:** They require a fixed input size, but sentences and audio clips vary in length.
* **No Feature Sharing:** If the network learns that "Harry" is a name at the start of a sentence, a standard network doesn't automatically apply that knowledge if "Harry" appears at the end.
* **Parameter Explosion:** Using a massive input layer for a long sequence of one-hot vectors leads to an unmanageable number of parameters.

### The Recurrent Neural Network (RNN) Architecture

An RNN processes data sequentially (e.g., left to right), maintaining a "hidden state" (activation) that acts as a memory of previous inputs.

* **The Hidden State ($a^{\langle t \rangle}$):** At each time step $t$, the network takes the current input $x^{\langle t \rangle}$ and the activation from the *previous* step $a^{\langle t-1 \rangle}$ to produce the new activation $a^{\langle t \rangle}$.
* **Recurrence and Shared Parameters:** The word "recurrent" applies to the weights. In a standard deep network, every layer has its own unique set of weights. In an RNN, the same weight matrices ($W_{aa}$, $W_{ax}$, $W_{ya}$) are used at every single time step, which drastically reduces the parameter count and allows for feature sharing. Also, because the same operation is applied "recurrently" to every word in a sentence or every frame in a video, the network is able to generalize patterns regardless of where they appear in the sequence.
* **Initialization:** The first activation ($a^{\langle 0 \rangle}$) is typically initialized as a vector of zeros.


### Forward Propagation in RNN

Forward propagation in a basic RNN follows these equations:

1. **Hidden State Update:**

$$a^{\langle t \rangle} = g_1(W_{aa} a^{\langle t-1 \rangle} + W_{ax} x^{\langle t \rangle} + b_a)$$


* $g_1$ is usually a **tanh** (or sometimes ReLU) activation function.


2. **Output Prediction:**

$$\hat{y}^{\langle t \rangle} = g_2(W_{ya} a^{\langle t \rangle} + b_y)$$


* $g_2$ is often **Sigmoid** (for binary classification like NER) or **Softmax**.


<div style='display: flex; justify-content: center'>
    <img src='images/rnn_forward.png' width=750px>
</div>

<br>

### Simplified Notation

To streamline implementation, we can stack the weight matrices $W_{aa}$ and $W_{ax}$ into a single matrix $W_a$:


$$a^{\langle t \rangle} = g(W_a [a^{\langle t-1 \rangle}, x^{\langle t \rangle}] + b_a)$$


Where $[a^{\langle t-1 \rangle}, x^{\langle t \rangle}]$ is a vector formed by stacking the previous activation on top of the current input.

### Limitations of The Basic RNN

The basic RNN is **unidirectional**, meaning the prediction at time $t$ only knows about the current and previous inputs.

* **Example:** In "He said, 'Teddy Roosevelt was a great president,'" and "He said, 'Teddy bears are on sale,'" the word "Teddy" can only be correctly classified if the model sees the words *after* it.
* **Solution:** This limitation is eventually solved by **Bidirectional RNNs (BRNNs)**, which we will cover in later sections.

---

## Backpropagation Through Time

This section touches on the training mechanism for RNN, specifically focusing on how gradients are calculated using **Backpropagation Through Time (BPTT)**.

### Key Concepts: The BPTT Framework

* **Computational Graph:** To train an RNN, we visualize it as a large unrolled graph. Forward propagation flows from left to right (increasing time $t$), while backpropagation flows from right to left (decreasing time $t$).
* **The Loss Function:** 
    * **Element-wise Loss ($L^{\langle t \rangle}$):** For each time step, a loss is calculated (usually Cross-Entropy Loss for classification) comparing the prediction $\hat{y}^{\langle t \rangle}$ to the ground truth $y^{\langle t \rangle}$.
    * **Total Loss ($L$):** The overall loss for the entire sequence is the sum of the losses at every time step:

    $$L = \sum_{t=1}^{T_y} L^{\langle t \rangle}(\hat{y}^{\langle t \rangle}, y^{\langle t \rangle})$$

### Backpropagation Through Time (BPTT)

It is called "Backpropagation Through Time" because the recursive chain rule calculations follow the hidden state connections back through the temporal sequence.

* **Direction of Flow:** Gradients are calculated by moving "backward in time." To calculate the gradient at time $t=1$, the network must account for the loss at that step *and* how its activation influenced all subsequent time steps ($t=2, 3, \dots, T_x$).
* **Parameter Updates:** Because the weights ($W_a, b_a, W_y, b_y$) are shared across all time steps, the final gradient for each parameter is the sum of the gradients calculated at every position in the sequence.

<br>

<div style='display: flex; justify-content: center'>
    <img src='images/rnn_backprop.png' width=750px>
</div>

<br>

### The Optimization Loop

1. **Forward Pass:** Calculate $a^{\langle t \rangle}$ and $\hat{y}^{\langle t \rangle}$ for all $t$ from $1$ to $T_x$.
2. **Loss Calculation:** Compute the total sequence loss $L$.
3. **Backward Pass (BPTT):** Compute derivatives of $L$ with respect to all parameters by scanning backward from $T_x$ to $1$.
4. **Update:** Use Gradient Descent (or Adam) to update the shared weight matrices.

---

<div style='color:red'>
    Backprop for RNN
</div>

---

## Different Types of RNNs

This section expands the basic RNN framework into a versatile family of architectures. Since $T_x$ (input length) and $T_y$ (output length) are often different in real-world applications, we use specific "many-to-many", "many-to-one", and "one-to-many" configurations to solve them.

### The RNN Architecture Family

The diversity of sequence problems is categorized by the relationship between the input and output lengths:

* **Many-to-Many ($T_x = T_y$):**
    * **Use Case:** Named Entity Recognition (NER).
    * **Logic:** Every input word $x^{\langle t \rangle}$ generates a corresponding output $y^{\langle t \rangle}$ immediately.
* **Many-to-One:**
    * **Use Case:** Sentiment Classification (e.g., movie reviews).
    * **Logic:** The network reads the entire sequence of words and only produces an output (like a 1–5 star rating) after the final input $x^{\langle T_x \rangle}$ has been processed.
* **One-to-Many:**
    * **Use Case:** Music Generation or Image Captioning.
    * **Logic:** A single input (like a genre code or an image) triggers the network to generate a sequence of outputs over multiple time steps.
* **Many-to-Many ($T_x \neq T_y$):**
    * **Use Case:** Machine Translation (e.g., French to English).
    * **Logic:** This is also known as an **Encoder-Decoder** architecture. The "Encoder" reads the whole input sequence first, and then the "Decoder" starts generating the output sequence in a different language.

<br>

<div style='display:flex; justify-content: center'>
    <img src='images/rnn_types.png' width=850px>
</div>

<br>

### Encoder-Decoder Deep Dive

The "Many-to-Many" version where $T_x \neq T_y$ is the most important for complex NLP.

1. **Encoder:** Processes the input sequence into a single "context vector" (the final hidden state).
2. **Decoder:** Uses that context vector as its initial state to begin generating a new sequence. This allows the model to "understand" the whole sentence before it starts translating, which is crucial for languages with different word orders.

### Summary Table of Architectures

| Type | Example Application | Input / Output Relationship |
| --- | --- | --- |
| **One-to-One** | Standard Neural Network | Non-sequential data. |
| **One-to-Many** | Music Generation | One prompt $\rightarrow$ sequence of notes. |
| **Many-to-One** | Sentiment Analysis | Sequence of words $\rightarrow$ one rating. |
| **Many-to-Many ($T_x = T_y$)** | Name Entity Recognition | Word $\rightarrow$ Label (simultaneous). |
| **Many-to-Many ($T_x \neq T_y$)** | Machine Translation | Full sentence $\rightarrow$ Full translation. |


### Next Step

The next technical hurdle is **Sequence Generation**, which involves "sampling" the output of one time step and feeding it as the input to the next.

---

## Language Models and Sequence Generation

This section explains the fundamental mechanics of **Language Modeling**, a cornerstone of NLP that enables machines to estimate the probability of a sequence of words.

### What is a Language Model?

The goal is to calculate probability of a sequence of words in a sentence, $P(y^{\langle 1 \rangle}, y^{\langle 2 \rangle}, \dots, y^{\langle T_y \rangle})$. This tells a system how likely a specific sentence is to occur in the real world compared to other similar-sounding sequences.
* Language model helps systems distinguish between homophones. For example, it helps a computer realize that "apple and **pear**" is statistically much more likely than "apple and **pair** (speech recognition utility)".
* In language models, vocabulary is a pre-defined dictionary of words or tokens (e.g., 10,000 words). In this vocabulary, we have special tokens. For examople `<EOS>` is a special "End of Sentence" token used to explicitly model where a thought concludes. `<UNK>` is the "Unknown" token used to replace any word not found in the vocabulary (e.g., a rare cat breed like "Mau").

### How the RNN Language Model Works

The architecture follows a specific "look-back" pattern where the output of one step becomes the input for the next.

* **Step 1 ($t=1$):** Input $x^{\langle 1 \rangle}$ is a zero vector. The model outputs a **Softmax** distribution $\hat{y}^{\langle 1 \rangle}$, representing the probability of every word in the dictionary being the *first* word.
* **Step 2 ($t=2$):** We set the input $x^{\langle 2 \rangle} = y^{\langle 1 \rangle}$ (the actual first word from the training set). The model then predicts the second word, given the first.
* **Step $t$:** The input $x^{\langle t \rangle}$ is always the ground-truth word from the previous time step $y^{\langle t-1 \rangle}$.
* **The Hidden State ($a^{\langle t \rangle}$):** This "memory" allows the network to maintain the context of *all* preceding words when predicting the current one.

### Mathematical Training & Inference

* **Loss Function:** For each word $t$, the model calculates a **Cross-Entropy Loss** ($L^{\langle t \rangle}$) between its prediction $\hat{y}^{\langle t \rangle}$ and the actual word $y^{\langle t \rangle}$.
* **Total Loss:** $L = \sum_t L^{\langle t \rangle}$.
* **Sentence Probability:** To find the probability of a full sentence like *"Cats average 15 hours"*, the model calculates:

$$P(\text{sentence}) = P(y^{\langle 1 \rangle}) \cdot P(y^{\langle 2 \rangle} \mid y^{\langle 1 \rangle}) \cdot P(y^{\langle 3 \rangle} \mid y^{\langle 1 \rangle}, y^{\langle 2 \rangle}) \dots$$

<br>
<div style='display:flex; justify-content:center'>
    <img src='images/rnn_lm.png' width=750px>
</div>
<br>

### Summary of the Data Flow

| Step | Input ($x$) | Output ($\hat{y}$) | Objective |
| --- | --- | --- | --- |
| **Start** | $\vec{0}$ | Probabilities for 1st word | Find most likely opener. |
| **Middle** | Actual word at $t-1$ | Probabilities for $t$ | Predict $t$ based on history. |
| **End** | Final word | Probabilities for `<EOS>` | Determine if sentence is over. |

---

## Sampling Novel Sequences

This section explains how to use a trained Recurrent Neural Network (RNN) to **sample** novel sequences. This process allows us to move from a model that simply "understands" patterns to one that can "generate" creative content like news articles or Shakespearean prose.

### Key Concepts: Sampling in Sequence Models

Sampling is a generative process where the model creates a sequence one step at a time.

* **Initialization:** Start with $a^{\langle 0 \rangle} = \vec{0}$ and $x^{\langle 1 \rangle} = \vec{0}$.
* **Softmax Sampling:** At each time step $t$, the model outputs a probability distribution ($\hat{y}^{\langle t \rangle}$). We randomly sample from this distribution (using a function like `np.random.choice`) to pick the actual word or character.
* **The Feedback Loop:** The word we just sampled becomes the input for the next time step ($x^{\langle t+1 \rangle} = \text{sampled } \hat{y}^{\langle t \rangle}$).
* **Termination:** Generation stops when the model samples an `<EOS>` (End of Sentence) token or reaches a predefined length limit (e.g., 100 words).

### Word-Level vs. Character-Level Models

| Feature | **Word-Level RNN** | **Character-Level RNN** |
| --- | --- | --- |
| **Vocabulary** | Large (e.g., 10,000+ words). | Very Small (A-Z, 0-9, space, punctuation). |
| **Unknowns** | Uses `<UNK>` for rare words. | No `<UNK>` needed; can construct any word letter-by-letter. |
| **Sequence Length** | Relatively short (10–20 steps). | Much longer (dozens or hundreds of steps). |
| **Pros** | Faster to train; captures long dependencies. | Handles any vocabulary; no "unknown word" errors. |
| **Cons** | Limited by dictionary size. | Computationally expensive; struggles with long-range context. |

### Implementation & Performance Notes

* **Mimicking Style:** The model adopts the "personality" of its training data. A model trained on news will generate journalistic snippets, while one trained on Shakespeare will produce archaic, poetic text.
* **Refining Output:** To improve quality, we can explicitly program the sampler to "reject" and resample if it picks an `<UNK>` token.
* **The Vanishing Gradient Problem:** A major limitation of basic RNNs is their difficulty in remembering the beginning of a long sequence (e.g., matching a subject at the start with a verb at the end). This leads to the need for more advanced units like **GRUs** (Gated Recurrent Units) or **LSTMs** (Long Short Term Memory), which will be covered later.

---

## Vanishing Gradients with RNNs

This section highlights the fundamental training difficulties of standard RNNs, primarily focusing on why they struggle to remember information over long distances within a sequence.

### Gradient Challenges in RNNs

* **Long-Term Dependencies:** In natural language, a word at the beginning of a sentence often dictates a choice much later.
    * *Example:* "**The cat** [long descriptive clause] **was** full" vs. "**The cats** [long descriptive clause] **were** full."
    * The model must "remember" the singular or plural status of the subject across many time steps.
* **The Vanishing Gradient Problem:** 
    * An RNN unrolled over 1,000 time steps is essentially a 1,000-layer deep neural network.
    * During backpropagation, the gradient (error signal) often decreases exponentially as it travels back to earlier time steps.
    * Consequently, the model becomes "short-sighted." The weights at $t=1$ are rarely updated based on errors occurring at $t=100$, making it nearly impossible to learn long-range relationships.
* **Basic RNN fails at Memory:** In a standard RNN, the hidden state is constantly being overwritten by new inputs. Because the gradient vanishes, the network lacks a mechanism to "lock in" a specific piece of information (like "singular subject") and preserve it while ignoring irrelevant middle-sentence fluff.

### Vanishing vs. Exploding Gradients

Like in deep CNN, the root cause of vanishing or exploding gradients in RNN architectures is the recursive application of the chain rule. In deep CNNs, this happens across spatial layers; in RNNs, it occurs across temporal steps.The table below summarizes the symptons and primary solutions of vanishing and exploding gradients in deep RNN and CNN architectures:

| Problem | Symptom | Severity in RNNs | Primary Solutions |
| --- | --- | --- | --- |
| **Exploding Gradients** | Gradients grow exponentially; weight updates become unstable, often resulting in `NaN` overflows. | High; easy to detect but immediately halts training. | **Gradient Clipping** (global or by-norm); **Batch Normalization** (in CNNs). |
| **Vanishing Gradients** | Gradients decay exponentially; early time steps or layers stop updating, losing long-range context. | **Critical**; the primary bottleneck for standard RNNs. Very subtle to diagnose. | **Gated Units** (GRUs, LSTMs); **Residual Connections** (CNNs); **ReLU** activations. |


### Detection of Exploding or Vanishing Gradients in RNN

To debug a training loop:

1. **Exploding:** We monitor the **$\ell_2$ norm** of the gradients. If it spikes or we see `NaN`, we implement `clipnorm` or `clipvalue` in our optimizer.
2. **Vanishing:** We look for a **flat loss curve** that doesn't move from the start, or check if the weights in our first layers are stagnant compared to our last layers.

### Summary

Exploding gradients can be effectively handled using **Gradient Clipping**. However, it takes more sophosticated solutions to handle vanishing gradients in deep RNN. These solutions are as follow, which we will cover in next sections:

1. **GRU (Gated Recurrent Unit):** Uses "gates" to decide when to keep or drop information.
2. **LSTM (Long Short-Term Memory):** An even more robust version of the gated architecture.

---

## Gated Recurrent Unit (GRU)

The **Gated Recurrent Unit (GRU)** is a powerful modification of the standard RNN hidden layer. It was specifically designed to handle long-range dependencies and solve the vanishing gradient problem, allowing the network to remember information across hundreds of time steps. Research by [Cho et al. (2014)](https://arxiv.org/abs/1409.1259) and [Chung et al. (2014)](https://arxiv.org/abs/1412.3555) popularized the GRU as a simpler, faster alternative to the LSTM (see next section).

### The Memory Cell

The defining feature of a GRU is the **Memory Cell ($c_t$)**, which is comprised of two logic gates to remember or forget previous information (hidden layer).

* In a GRU, the cell value is exactly equal to the output activation ($c_t = a_t$).
* Its job is to provide a "persistent" bit of memory. For example, it might store whether a subject is singular ("Cat") or plural ("Cats") so that the model chooses the correct verb ("was" vs "were") much later in a sentence.

### The Four Equations of a GRU

A GRU uses two "gates" to manage how information is stored and forgotten. These gates are typically vectors of the same dimension as the hidden state, allowing for element-wise control.

#### 1. The Update Gate ($\Gamma_u$)

This gate decides whether the memory cell should be updated with new information or keep its old value.

$$\Gamma_u = \sigma(W_u [c_{t-1}, x_t] + b_u)$$

* **Intuition:** Think of this as a value between $0$ and $1$. If $\Gamma_u \approx 0$, the model "hangs on" to the old memory. If $\Gamma_u \approx 1$, it "updates" to a new concept.

#### 2. The Reset (Relevance) Gate ($\Gamma_r$)

This gate determines how much of the *previous* memory is relevant to calculating the *next* potential memory candidate.

$$\Gamma_r = \sigma(W_r [c_{t-1}, x_t] + b_r)$$

#### 3. The Candidate Cell ($\tilde{c}_t$)

This is the "suggestion" for what the new memory should be. Note how the reset gate ($\Gamma_r$) can pull in or ignore the past memory.

$$\tilde{c}_t = \tanh(W_c [\Gamma_r * c_{t-1}, x_t] + b_c)$$

#### 4. The Final Cell Update ($c_t$)

This is the most critical equation. It performs a linear interpolation between the old state and the new candidate.

$$c_t = \Gamma_u * \tilde{c}_t + (1 - \Gamma_u) * c_{t-1}$$

### Why GRUs Solve Vanishing Gradients

In a standard RNN, gradients must pass through complex weight multiplications at every step, causing them to shrink.

* In a GRU, if the Update Gate ($\Gamma_u$) is very close to $0$, the network effectively sets $c_t = c_{t-1}$.
* This creates a "linear highway" where the memory value stays nearly constant. Because the value is maintained exactly, the gradient can flow back through time without disappearing.

### Summary Comparison

| Component | Function | Intuition |
| --- | --- | --- |
| **Update Gate ($\Gamma_u$)** | Control persistence | "Should I remember the subject of this sentence?" |
| **Reset Gate ($\Gamma_r$)** | Control relevance | "How much of the past matters for this new word?" |
| **Memory Cell ($c_t$)** | Information storage | The actual data (e.g., "Singular Subject") being held. |

### Performanec and Application
* On most tasks, GRUs perform just as well as LSTMs but are more computationally efficient because they have fewer parameters (two gates instead of three).
* They are the "standard" choice for smaller datasets or when hardware resources are limited.

### Common Alternative Notations

In academic papers and deep learning libraries (like PyTorch or Keras), the notation for GRUs can vary significantly. While the "Gamma" ($\Gamma$) notation is excellent for learning because it emphasizes the "Gate," researchers often use more compact variable names.

The most frequent mapping we will see in the literature (e.g., in the original paper by Cho et al.) is as follows:

| Concept | "Learning" Notation | "Literature" Notation |
| --- | --- | --- |
| **Update Gate** | $\Gamma_u$ | $z_t$ |
| **Reset Gate** | $\Gamma_r$ | $r_t$ |
| **Candidate State** | $\tilde{c}_t$ | $\tilde{h}_t$ or $\hat{h}_t$ |
| **Final Hidden State** | $c_t$ or $a_t$ | $h_t$ |

Also, here are the GRU equations written in altenative notations:

1. **Reset Gate:** $r_t = \sigma(W_r \cdot [h_{t-1}, x_t])$
2. **Update Gate:** $z_t = \sigma(W_z \cdot [h_{t-1}, x_t])$
3. **Candidate Hidden State:** $\tilde{h}_t = \tanh(W \cdot [r_t \odot h_{t-1}, x_t])$
4. **Final Hidden State:** $h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$

> **Note on Symbols:** Researchers often use the symbol $\odot$ (the Hadamard product) to represent element-wise multiplication instead of an asterisk ($*$).

Here is the depiction of a GRU cell borrowed from [here](https://towardsdatascience.com/gru-recurrent-neural-networks-a-smart-way-to-predict-sequences-in-python-80864e4fe9f6/)

<div style='display: flex; justify-content:center'>
    <img src='images/gru.png' width=500px>
</div>

<br>

To build an intuition on how the GRU works, watch [this short video](https://www.youtube.com/watch?v=IBs8D8PWMc8).

---

## Long Short Term Memory (LSTM)

The **Long Short-Term Memory (LSTM)** unit is a seminal architecture in sequence modeling, developed by Hochreiter and Schmidhuber. It is a more powerful and generalized version of the GRU, specifically engineered to maintain information over very long time steps by using a more complex system of gates.

### LSTM Key Concepts

While the GRU is a simplified, more recent invention, the LSTM has been the "historically proven" standard for decades. The most significant structural difference is that the LSTM separates the Hidden State ($a_t$) from the Memory Cell ($c_t$), whereas in a GRU, they are the same.

* **Three Gates:** Instead of two, the LSTM uses an Update gate, a Forget gate, and an Output gate.
* **Decoupled Updates:** In a GRU, the update gate $\Gamma_u$ and forget gate $(1 - \Gamma_u)$ are tied. In an LSTM, the model can choose to forget the past *without* necessarily adding new information, or vice-versa.

### The LSTM Equations

The behavior of an LSTM is governed by these primary calculations:

1. **Candidate Value ($\tilde{c}_t$):** A $\tanh$ layer that creates a vector of new candidate values that could be added to the state.
2. **Update Gate ($\Gamma_u$):** Decides which values we’ll update.
3. **Forget Gate ($\Gamma_f$):** Decides what information we’re going to throw away from the cell state.
4. **Output Gate ($\Gamma_o$):** Decides what part of the cell state we are going to output.

$$\tilde{c}_t = tanh(W_c[a_{t-1}, x_t] + b_c)$$

$$\Gamma_u = \sigma(W_u[a_{t-1}, x_t] + b_u)$$

$$\Gamma_f = \sigma(W_f[a_{t-1}, x_t] + b_f)$$

$$\Gamma_o = \sigma(W_o[a_{t-1}, x_t] + b_o)$$

$$c_t = \Gamma_u * \tilde{c}_t + \Gamma_f * c_{t-1}$$

$$a_t = \Gamma_o * \tanh(c_t)$$

<br>

<div style='display: flex; justify-content: center'>
    <img src='images/lstm.png' width=500px>
</div>

### Comparison Summary

| Feature | **GRU** | **LSTM** |
| --- | --- | --- |
| **Complexity** | 2 Gates (Update, Reset) | 3 Gates (Update, Forget, Output) |
| **Memory Flow** | $a_t = c_t$ | Separate $a_t$ and $c_t$ |
| **Speed** | Faster (fewer parameters) | Slower (more computation) |
| **Flexibility** | Good for scaling to very large models | More powerful and flexible for complex dependencies |
| **Default Choice** | Modern favorite for efficiency | Historically proven default for most tasks |

### Advanced Variations

* **Peephole Connections:** A common variation where the gate layers ($u, f, o$) look not only at the previous hidden state $a_{t-1}$ and input $x_t$, but also at the previous memory cell value $c_{t-1}$. This allows the gates to be even more precisely tuned to the current state of the memory.
* **Dimensionality:** In practice, these are high-dimensional vectors (e.g., 100D). Each "bit" in the memory cell can represent a different concept (singular/plural, gender, presence of a quote, etc.), and the gates act element-wise on these specific features.

### Recommendation for Implementation

If we are deciding between the two:

* **Try LSTM first** if we want the most robust, proven architecture and have the computational budget.
* **Switch to GRU** if we need to build much larger models, require faster training cycles, or have a smaller dataset where the LSTM might overfit.

<br>

To build an intuition on how the GRU works, watch [this short video](https://www.youtube.com/watch?v=VJHWOO2Ix_8).